In [15]:
import requests
import pandas as pd
import time
from typing import Tuple
import json

In [ ]:
# 配置DeepSeek API
api_key = "sk-xxxxxxxxxxxxxxxxxxxxxxx"  # 替换为您的API密钥
api_url = "https://api.deepseek.com"

In [20]:
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "你是谁"},
    ],
    stream=False
)

print(response.choices[0].message.content)

我是DeepSeek Chat，由深度求索公司创造的智能AI助手！✨ 我的使命是帮助你解答各种问题，无论是学习、工作，还是日常生活中的小困惑，我都会尽力提供准确、有用的信息。有什么我可以帮你的吗？😊


In [25]:
# 定义分类标签
categories =  {
    "农业": {
        "种植业": ["小麦", "水稻", "玉米", "大豆", "蔬菜", "水果", "棉花"],
        "林业": ["木材", "造林", "经济林", "竹木", "林产品"],
        "畜牧业": ["养猪", "养牛", "养鸡", "畜牧", "饲料"],
        "副业": ["养蜂", "蚕桑", "药材", "人参", "食用菌"],
        "渔业": ["捕鱼", "水产", "养殖鱼", "海产品"]
    },
    "工业": {
        "采矿": ["煤矿", "铁矿", "石油", "采石", "金属矿"],
        "制造": ["农具", "机械", "纺织", "食品加工", "家具"],
        "电力": ["发电", "电网", "新能源", "电力供应"],
        "水生产": ["自来水", "污水处理", "供水系统"],
        "建筑": ["房屋建设", "道路修建", "水利工程"]
    },
    "第三产业": {
        "水利": ["灌溉", "防洪", "水资源管理"],
        "研究": ["农业科研", "技术推广", "实验室"],
        "教育": ["乡村学校", "职业培训", "文化站"],
        "社会保障": ["医疗服务", "养老院", "福利设施"]
    }
}

In [26]:
# 创建一个文本分类函数
def classify_text(text):
    # 分词后的文本数据
    data = {
        'text': text
    }

    # 设置请求头，通常需要设置Authorization字段来传递API密钥
    headers = {
        'Authorization': f'Bearer {api_key}'  # 使用Bearer认证
    }

    # 调用API进行文本分类
    response = requests.post(api_url, json=data, headers=headers)
    response_data = response.json()

    # 假设返回的结果包括主分类和子分类
    main_category = response_data.get('main_category', '')
    sub_category = response_data.get('sub_category', '')

    # 检查是否属于二级分类
    if main_category in categories:
        if sub_category in categories[main_category]:
            return f"{text} 被分类为 {main_category} 的 {sub_category}"
        else:
            return f"{text} 被分类为 {main_category}，但未能匹配到子类"
    else:
        return f"{text} 未匹配到任何分类"

# 测试样本
sample_text = '小麦'

# 获取分类结果
result = classify_text(sample_text)
print(result)

小麦 未匹配到任何分类


In [9]:
# 分类体系定义（全局变量）
CATEGORY_HIERARCHY = {
    "农业": {
        "种植业": ["小麦", "水稻", "玉米", "大豆", "蔬菜", "水果", "棉花"],
        "林业": ["木材", "造林", "经济林", "竹木", "林产品"],
        "畜牧业": ["养猪", "养牛", "养鸡", "畜牧", "饲料"],
        "副业": ["养蜂", "蚕桑", "药材", "人参", "食用菌"],
        "渔业": ["捕鱼", "水产", "养殖鱼", "海产品"]
    },
    "工业": {
        "采矿": ["煤矿", "铁矿", "石油", "采石", "金属矿"],
        "制造": ["农具", "机械", "纺织", "食品加工", "家具"],
        "电力": ["发电", "电网", "新能源", "电力供应"],
        "水生产": ["自来水", "污水处理", "供水系统"],
        "建筑": ["房屋建设", "道路修建", "水利工程"]
    },
    "第三产业": {
        "水利": ["灌溉", "防洪", "水资源管理"],
        "研究": ["农业科研", "技术推广", "实验室"],
        "教育": ["乡村学校", "职业培训", "文化站"],
        "社会保障": ["医疗服务", "养老院", "福利设施"]
    }
}

In [10]:
def format_category_hierarchy() -> str:
    """将分类体系格式化为字符串用于prompt"""
    desc = []
    for main, subs in CATEGORY_HIERARCHY.items():  # 现在可以正确访问全局变量
        sub_desc = [f"{sub}（示例：{', '.join(examples[:3])}等）" 
                   for sub, examples in subs.items()]
        desc.append(f"- {main}: {'; '.join(sub_desc)}")
    return "\n".join(desc)

def get_hierarchical_classification(word: str) -> Tuple[str, str]:
    """使用DeepSeek API进行两级分类"""
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    
    # 构造分类prompt（现在可以调用format_category_hierarchy()）
    prompt = f"""
    请将以下生产活动词语按二级分类体系分类，格式为"主类-子类"：
    
    分类体系：
    {format_category_hierarchy()}
    
    示例：
    - 小麦 → 农业-种植业
    - 养猪 → 农业-畜牧业
    - 农具 → 工业-制造
    - 学校 → 第三产业-教育
    
    只需返回分类结果，不要解释。如果无法确定子类，则返回主类。
    
    需要分类的词语：{word}
    """
    
    payload = {
        "model": "deepseek-chat",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1,
        "max_tokens": 20
    }
    
    try:
        response = requests.post(API_URL, headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()["choices"][0]["message"]["content"].strip()
        
        if "-" in result:
            main, sub = result.split("-", 1)
            return main.strip(), sub.strip()
        return result.strip(), "未指定子类"
    
    except Exception as e:
        print(f"分类错误: {word} - {str(e)}")
        return "分类失败", ""

In [11]:
def format_category_hierarchy() -> str:
    """将分类体系格式化为字符串用于prompt"""
    desc = []
    for main, subs in CATEGORY_HIERARCHY.items():
        sub_desc = [f"{sub}（示例：{', '.join(examples[:3])}等）" 
                   for sub, examples in subs.items()]
        desc.append(f"- {main}: {'; '.join(sub_desc)}")
    return "\n".join(desc)

In [12]:
def local_classify(word: str) -> Tuple[str, str]:
    """本地规则分类器（备用方案）"""
    # 这里可以添加更详细的本地规则
    # 示例简单实现，实际应扩展更多关键词
    for main, subs in CATEGORY_HIERARCHY.items():
        for sub, keywords in subs.items():
            if any(kw in word for kw in keywords):
                return main, sub
    return "第三产业", "其他服务"

def batch_classify(words: list, use_api: bool = True) -> pd.DataFrame:
    """批量分类处理"""
    results = []
    for word in words:
        if use_api:
            main, sub = get_hierarchical_classification(word)
        else:
            main, sub = local_classify(word)
        
        results.append({
            "词语": word,
            "主类": main,
            "子类": sub,
            "完整分类": f"{main}-{sub}" if sub else main
        })
        time.sleep(0.3)  # API请求间隔
    
    return pd.DataFrame(results)

In [13]:
# 示例使用
if __name__ == "__main__":
    # 测试数据
    test_words = [
        "小麦", "养猪", "木材加工", "水产养殖", 
        "农具制造", "电力供应", "乡村学校", "水利工程"
    ]
    
    print("开始分类...")
    df = batch_classify(test_words)
    
    print("\n分类结果:")
    print(df[["词语", "完整分类"]])
    
    # 保存完整结果
    df.to_excel("生产活动二级分类.xlsx", index=False)
    print("\n结果已保存到: 生产活动二级分类.xlsx")

开始分类...
分类错误: 小麦 - 404 Client Error: Not Found for url: https://api.deepseek.com/
分类错误: 养猪 - 404 Client Error: Not Found for url: https://api.deepseek.com/
分类错误: 木材加工 - 404 Client Error: Not Found for url: https://api.deepseek.com/
分类错误: 水产养殖 - 404 Client Error: Not Found for url: https://api.deepseek.com/
分类错误: 农具制造 - 404 Client Error: Not Found for url: https://api.deepseek.com/
分类错误: 电力供应 - 404 Client Error: Not Found for url: https://api.deepseek.com/
分类错误: 乡村学校 - 404 Client Error: Not Found for url: https://api.deepseek.com/
分类错误: 水利工程 - 404 Client Error: Not Found for url: https://api.deepseek.com/

分类结果:
     词语  完整分类
0    小麦  分类失败
1    养猪  分类失败
2  木材加工  分类失败
3  水产养殖  分类失败
4  农具制造  分类失败
5  电力供应  分类失败
6  乡村学校  分类失败
7  水利工程  分类失败

结果已保存到: 生产活动二级分类.xlsx


In [14]:
import urllib.request
import socket

def check_network():
    try:
        # 测试直接连接
        urllib.request.urlopen("https://api.deepseek.com", timeout=3)
        print("✅ 直接连接成功")
        return True
    except Exception as e:
        print(f"❌ 直接连接失败: {str(e)}")
        
        # 测试DNS解析
        try:
            socket.gethostbyname("api.deepseek.com")
            print("✅ DNS解析正常")
        except socket.gaierror:
            print("❌ DNS解析失败（可能域名错误或网络封锁）")
        
        return False

check_network()

❌ 直接连接失败: HTTP Error 404: Not Found
✅ DNS解析正常


False

In [2]:
import requests

def verify_api_endpoint():
    test_urls = [
        "https://api.deepseek.com/v1/chat/completions",  # 常见RESTful路径
        "https://api.deepseek.com/chat/completions",     # 原路径
        "https://platform.deepseek.com/api/chat"         # 可能的备用路径
    ]
    
    for url in test_urls:
        try:
            response = requests.head(url, timeout=5)
            print(f"{url} -> HTTP {response.status_code}")
        except Exception as e:
            print(f"{url} -> 错误: {str(e)}")

verify_api_endpoint()

https://api.deepseek.com/v1/chat/completions -> HTTP 401
https://api.deepseek.com/chat/completions -> HTTP 401
https://platform.deepseek.com/api/chat -> HTTP 429
